# Task 3 – LongEval-USim: Next Query Prediction

**Prompt A** = alter Prompt (nur Queries)  
**Prompt B** = neuer Prompt (Queries + angeklickte Docs + ignorierte Docs)

| Run | query_mode | prompt_variant | run_name |
|-----|-----------|---------------|----------|
| 1 | `first` | `A` | `openai_first_promptA` |
| 2 | `last` | `A` | `openai_last_promptA` |
| 3 | `all` | `A` | `openai_all_promptA` |
| 4 | `first` | `B` | `openai_first_promptB` |
| 5 | `last` | `B` | `openai_last_promptB` |
| 6 | `all` | `B` | `openai_all_promptB` |

In [9]:
%pip install openai python-dotenv --quiet
%pip install git+https://github.com/clef-longeval/ir-datasets-longeval --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os, json, time, re, zipfile, ast
import pandas as pd
from pathlib import Path
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / '.env').exists():
        load_dotenv(folder / '.env', override=True)
        break

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '').strip().strip('"').strip("'")
if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY fehlt!')
client = OpenAI(api_key=OPENAI_API_KEY)
print('OpenAI Client bereit.')

OpenAI Client bereit.


In [11]:
# !! NUR DIESE 3 WERTE AENDERN fuer jeden Run !!
RUN_CONFIG = {
    'query_mode':     'all',               # 'first' | 'last' | 'all'
    'prompt_variant': 'B',                   # 'A' = nur Queries | 'B' = mit Docs
    'run_name':       'openai_all_promptB',
    # ab hier nichts aendern
    'model':         'gpt-4o-mini',
    'team_name':     'JOINorDIE',
    'description':   'LLM-based next query prediction using OpenAI GPT-4o-mini.',
    'max_doc_chars':  600,
    'n_rel_docs':     3,
    'n_nonrel_docs':  3,
    'output_dir':    'runfiles_task3',
}

SNAPSHOT_FILES = {
    'snapshot-1.jsonl': 'predetermined_queries_Task_A_test.csv',
    'snapshot-2.jsonl': 'task3_longeval_usim-sessions-06-08_2025.csv',
    'snapshot-3.jsonl': 'task3_longeval_usim-sessions-09-11_2025.csv',
}

os.makedirs(RUN_CONFIG['output_dir'], exist_ok=True)
print('Run:', RUN_CONFIG['run_name'], '| Prompt:', RUN_CONFIG['prompt_variant'])

Run: openai_all_promptB | Prompt: B


In [12]:
# Dokument-Lookup (nur fuer Prompt B) + vorhandene Topics laden
doc_lookup = {}

if RUN_CONFIG['prompt_variant'] == 'B':
    from ir_datasets_longeval import load as ld_load
    print('Lade Dokument-Lookup (~5-10 Min, einmalig)...')
    dataset = ld_load('longeval-sci-2026/snapshot-1/train')
    for i, doc in enumerate(dataset.docs_iter()):
        doc_lookup[str(doc.doc_id)] = {
            'title':    getattr(doc, 'title', '') or '',
            'abstract': getattr(doc, 'abstract', '') or ''
        }
        if i % 300000 == 0 and i > 0:
            print(f'  {i:,} Docs geladen...')
    print(f'Fertig: {len(doc_lookup):,} Docs.')
else:
    print('Prompt A: kein Dokument-Lookup noetig.')

# Vorhandene Topics aus topics_full_comparison_openai.csv laden
# (35 Training-Sessions mit title, desc, narr, tfidf_terms)
import pandas as pd
topic_lookup = {}
try:
    topics_df = pd.read_csv('topics_full_comparison_openai.csv')
    for _, row in topics_df.iterrows():
        sid = str(row['session_id'])
        topic_lookup[sid] = {
            'title':       str(row.get('title_all', row.get('title_first', ''))).strip(),
            'description': str(row.get('desc_all',  row.get('desc_first',  ''))).strip(),
            'narrative':   str(row.get('narr_all',  row.get('narr_first',  ''))).strip(),
            'tfidf_terms': str(row.get('tfidf_terms_all', row.get('tfidf_terms_first', ''))).strip(),
        }
    print(f'Topics geladen: {len(topic_lookup)} Sessions aus topics_full_comparison_openai.csv')
except FileNotFoundError:
    print('topics_full_comparison_openai.csv nicht gefunden - generiere on-the-fly')


Lade Dokument-Lookup (~5-10 Min, einmalig)...
  300,000 Docs geladen...
  600,000 Docs geladen...
Fertig: 869,902 Docs.
Topics geladen: 35 Sessions aus topics_full_comparison_openai.csv


In [13]:
def parse_docnos(val):
    try:
        return [str(x) for x in ast.literal_eval(str(val))]
    except:
        return []

def parse_interactions(val):
    try:
        items = ast.literal_eval(str(val))
        return [str(item[0]) for item in items if isinstance(item, (list, tuple))]
    except:
        return []

def load_sessions(csv_path):
    df = pd.read_csv(csv_path, header=None)
    df.columns = ['idx','user','session_id','query','timestamp',
                  'retrieved_docs','session_hash','interactions']
    sessions = defaultdict(lambda: {'queries': [], 'rel': set(), 'nonrel': set()})
    for _, row in df.sort_values('timestamp').iterrows():
        sid = str(row['session_id'])
        sessions[sid]['queries'].append(str(row['query']))
        retrieved  = parse_docnos(row['retrieved_docs'])
        interacted = set(parse_interactions(row['interactions']))
        sessions[sid]['rel'].update(interacted)
        sessions[sid]['nonrel'].update(d for d in retrieved if d not in interacted)
    return {
        sid: {'queries': d['queries'],
              'rel_docnos': list(d['rel']),
              'nonrel_docnos': list(d['nonrel'])}
        for sid, d in sessions.items()
    }

test = load_sessions('task3_longeval_usim-sessions-06-08_2025.csv')
sid0 = list(test.keys())[0]
print(f'Sessions: {len(test)}')
print(f'Session {sid0}: queries={test[sid0]["queries"]}')
print(f'  angeklickt={test[sid0]["rel_docnos"][:3]}')

Sessions: 119
Session 115: queries=['parenting style and identity of adolescents', "parent's identitystyle and identity of adolescents"]
  angeklickt=['82284189']


In [14]:
def get_context(queries, mode):
    ctx = queries[:-1] if len(queries) > 1 else queries
    if mode == 'first': return ctx[0] if ctx else queries[0]
    if mode == 'last':  return ctx[-1] if ctx else queries[0]
    return chr(10).join(ctx)

def get_doc_text(docno):
    d = doc_lookup.get(str(docno), {})
    t = (d.get('title', '') or '').strip()
    a = (d.get('abstract', '') or '').strip()
    text = ('Title: ' + t + chr(10) + 'Abstract: ' + a) if (t or a) else ('Document ID: ' + docno)
    return text[:RUN_CONFIG['max_doc_chars']]

# Cache fuer on-the-fly generierte Topics
topic_cache = {}

def get_topic(sid, all_queries):
    """Gibt Topic zurueck: erst aus vorhandenen Topics, dann on-the-fly generiert."""
    # 1. Aus topics_full_comparison_openai.csv (snapshot-1 Training)
    if sid in topic_lookup:
        return topic_lookup[sid]
    # 2. Aus Cache (schon on-the-fly generiert)
    if sid in topic_cache:
        return topic_cache[sid]
    # 3. On-the-fly generieren (snapshot-2 / snapshot-3)
    queries_text = chr(10).join(all_queries[:-1] if len(all_queries) > 1 else all_queries)
    prompt = (
        'You are an expert information retrieval analyst. '
        'Create a TREC-style search topic from these search queries. '
        'Return ONLY valid JSON with keys: title, description, narrative.' + chr(10)*2 +
        'Search queries: ' + queries_text
    )
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=RUN_CONFIG['model'],
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.3, max_tokens=300,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if match:
                data = json.loads(match.group())
                t = str(data.get('title', queries_text[:30])).strip()
                d = str(data.get('description', '')).strip()
                n = str(data.get('narrative', '')).strip()
                if d and n:
                    topic = {'title': t, 'description': d, 'narrative': n, 'tfidf_terms': ''}
                    topic_cache[sid] = topic
                    return topic
        except Exception as e:
            time.sleep(2 ** attempt)
    fallback = {'title': all_queries[0][:30], 'description': 'Research about ' + all_queries[0],
                'narrative': 'N/A', 'tfidf_terms': ''}
    topic_cache[sid] = fallback
    return fallback


# PROMPT A: Research-Assistant-Stil mit Topics + TF-IDF (wie altes Ollama-Notebook)
def build_prompt_A(sid, queries_text, all_queries):
    topic = get_topic(sid, all_queries)
    time.sleep(0.2)
    system = (
        'You are a precise research assistant. '
        'Your sole task is to generate follow-up search queries strictly based on the provided topic context. '
        'Rules:' + chr(10) +
        '- Use ONLY information explicitly present in the context below.' + chr(10) +
        '- Do NOT invent new topics, assumptions, or unrelated content.' + chr(10) +
        '- Be specific, neutral, and concise.' + chr(10) +
        '- Output ONLY valid JSON - no intro, no explanation, no commentary.'
    )
    tfidf = topic.get('tfidf_terms', '').strip() or 'N/A'
    user = (
        'Based on the research topic below, predict the 5 most likely next search queries ' +
        'that this researcher would naturally search for next.' + chr(10)*2 +
        'TOPIC CONTEXT:' + chr(10) +
        'Title:            ' + topic['title'] + chr(10) +
        'Description:      ' + topic['description'] + chr(10) +
        'Narrative:        ' + topic['narrative'] + chr(10) +
        'Previous Queries: ' + queries_text + chr(10) +
        'Key Terms:        ' + tfidf + chr(10)*2 +
        'Output Format (strictly):' + chr(10) +
        'Return ONLY valid JSON with key "queries": array of exactly 5 strings.' + chr(10) +
        'No extra text, no markdown - only the JSON object.'
    )
    return system + chr(10)*2 + user


# PROMPT B: Queries + angeklickte Docs + ignorierte Docs
def build_prompt_B(queries_text, rel_docnos, nonrel_docnos):
    rel_block = ''
    for i, d in enumerate(rel_docnos[:RUN_CONFIG['n_rel_docs']], 1):
        rel_block += '[Document ' + str(i) + ']' + chr(10) + get_doc_text(d) + chr(10)*2
    if not rel_block.strip():
        rel_block = '(no clicked documents in this session)'
    nonrel_block = ''
    for i, d in enumerate(nonrel_docnos[:RUN_CONFIG['n_nonrel_docs']], 1):
        nonrel_block += '[Document ' + str(i) + ']' + chr(10) + get_doc_text(d) + chr(10)*2
    if not nonrel_block.strip():
        nonrel_block = '(no skipped documents available)'
    sep = chr(10)
    return sep.join([
        'Given some user queries, relevant and not relevant documents,',
        'predict the 5 most likely next search queries the user will type.',
        '',
        'Queries',
        'A person has typed these queries into a search engine:',
        queries_text,
        '',
        '--- BEGIN RELEVANT DOCUMENTS CONTENT ---',
        rel_block.strip(),
        '--- END RELEVANT DOCUMENTS CONTENT ---',
        '',
        '--- BEGIN NOT RELEVANT DOCUMENTS CONTENT ---',
        nonrel_block.strip(),
        '--- END NOT RELEVANT DOCUMENTS CONTENT ---',
        '',
        'Output Format and Structure:',
        'Return ONLY valid JSON with key "queries": array of exactly 5 strings.',
        'Rank from most to least likely. Keep queries concise and on-topic.',
        'No extra text, no markdown - only the JSON object.',
    ])


def generate_next_queries(sid, sdata, retries=3):
    queries_text = get_context(sdata['queries'], RUN_CONFIG['query_mode'])
    if RUN_CONFIG['prompt_variant'] == 'A':
        prompt = build_prompt_A(sid, queries_text, sdata['queries'])
    else:
        prompt = build_prompt_B(queries_text, sdata['rel_docnos'], sdata['nonrel_docnos'])
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=RUN_CONFIG['model'],
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.5, max_tokens=400,
            )
            raw = resp.choices[0].message.content.strip()
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if match:
                data = json.loads(match.group())
                qs = data.get('queries', [])
                if isinstance(qs, list) and len(qs) >= 1:
                    return [str(q).strip() for q in qs[:5]]
        except Exception as e:
            print(f'  Session {sid}, Attempt {attempt+1}: {e}')
            time.sleep(2 ** attempt)
    last = sdata['queries'][-2] if len(sdata['queries']) > 1 else sdata['queries'][0]
    return [last, last+' review', last+' study', last+' analysis', last+' overview']


# Testlauf
print('Testlauf...')
t = generate_next_queries(sid0, test[sid0])
for i, q in enumerate(t, 1):
    print(f'  {i}. {q}')

Testlauf...
  1. impact of parenting styles on adolescent behavior
  2. types of parenting styles and their effects
  3. identity formation in teenagers
  4. how parenting influences adolescent identity
  5. parenting approaches and youth development


In [15]:
snapshot_results = {}

for snap_name, csv_path in SNAPSHOT_FILES.items():
    print(f'\n=== {snap_name} ===')
    sessions = load_sessions(csv_path)
    run = {'meta': {
        'team_name':   RUN_CONFIG['team_name'],
        'description': RUN_CONFIG['description'],
        'run_name':    RUN_CONFIG['run_name']
    }}
    for j, (sid, sdata) in enumerate(sessions.items(), 1):
        print(f'  [{j}/{len(sessions)}] {sdata["queries"][0][:60]}')
        run[sid] = generate_next_queries(sid, sdata)
        time.sleep(0.3)
    snapshot_results[snap_name] = run
    print(f'  -> {len(sessions)} Sessions fertig.')

print('\nAlle Snapshots verarbeitet!')


=== snapshot-1.jsonl ===
  [1/35] COMPARISON OF READERSHIP PERCEPTION OF ONLINE AND HARDCOPY N
  [2/35] passivation
  [3/35] multitasking
  [4/35] Global Aero Terminal 5320
  [5/35] ating a biodata
  [6/35] age and vocabulary acquisition strategis
  [7/35] FABM students and their academic performance
  [8/35] marine biological journal
  [9/35] Caucasus Refineries
  [10/35] Effects of working while studying among high school students
  [11/35] streaming services in nigeria
  [12/35] importance of identifying plant samples using pharmacognosti
  [13/35] effective sales strategy with content strategy on tiktok
  [14/35] lntelligence artificielle et audit
  [15/35]  Five Factor Inventory
  [16/35] Deep Learning for Tool Condition Monitoring
  [17/35] Metal injection molding
  [18/35] Middle English
  [19/35] The four day work week
  [20/35] Japanese national identity
  [21/35] Behaviour
  [22/35] general effects of cetirizine
  [23/35] psychoanalysis case study
  [24/35] digital marketing

In [16]:
output_zip = os.path.join(RUN_CONFIG['output_dir'],
    'task3_submission_' + RUN_CONFIG['run_name'] + '.zip')

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for snap_name, run_data in snapshot_results.items():
        zf.writestr(snap_name, json.dumps(run_data, ensure_ascii=False, indent=2))

errors = []
with zipfile.ZipFile(output_zip, 'r') as zf:
    print('Dateien im ZIP:', zf.namelist())
    for name in zf.namelist():
        data = json.loads(zf.read(name))
        sids = [k for k in data if k != 'meta']
        for key, val in data.items():
            if key == 'meta': continue
            if not isinstance(val, list) or len(val) < 1:
                errors.append(f'{name} Session {key}: keine Queries')
        print(f'  {name}: {len(sids)} Sessions')

if errors:
    print('FEHLER:', errors[:5])
else:
    print(f'\nSessions: ' + str({k: len(v)-1 for k,v in snapshot_results.items()}))
    print(f'\nGespeichert: {output_zip}')
    print('Auf TIRA unter task-3-run-upload hochladen!')

Dateien im ZIP: ['snapshot-1.jsonl', 'snapshot-2.jsonl', 'snapshot-3.jsonl']
  snapshot-1.jsonl: 35 Sessions
  snapshot-2.jsonl: 119 Sessions
  snapshot-3.jsonl: 182 Sessions

Sessions: {'snapshot-1.jsonl': 35, 'snapshot-2.jsonl': 119, 'snapshot-3.jsonl': 182}

Gespeichert: runfiles_task3\task3_submission_openai_all_promptB.zip
Auf TIRA unter task-3-run-upload hochladen!
